# Predicting IQ at age 3 from neonatal HRV

*Group A · Jade · PRI 2026 · AuBE cohort (Saint-Étienne University Hospital)* · *English version of `01_prediction_qit.ipynb`: same code, same numbers.*

**Question**: do heart rate variability (HRV) indices recorded at birth help predict the IQ measured around age 3 (WPPSI-III), beyond clinical factors? In Samuel Fornet's report, no model did better than predicting the mean IQ (MAE ≈ 10.3, R² ≤ 0).

| Plan for the Week | Section |
|---|---|
| **Tuesday** · one recording per child, complete scores, merge, cleaning | 1 |
| **Tuesday** · feature reduction: correlation > 0.95, near-zero variance, random forest, PCA | 2 |
| **Wednesday** · Elastic Net, Random Forest, LightGBM, XGBoost, SVR | 3 |
| **Thursday** · 10 × 5 cross-validation, QIT / QIV / QIP / NCL, MAE, MSE, RMSE, R², `model_comparison.csv` | 4 |
| **Why it adds nothing**, and from when it would have added something | 4.4 |
| **Bound**: what share of IQ can HRV explain, at most? | 4.5 |
| **Positive control**: does HRV recognise prematurity? | 5 |
| **Classification** FSIQ < 90 | 6 |
| **Hypotheses**: later HRV (M6 to M24) and trajectories | 7 |
| **Supervisor's advice** · conformal prediction, feature explanation | 8, 9 |
| **Friday** · summary for the supervisor (`SUMMARY.md`), data check | 10, 11 |

**Organisation**: this notebook shows the steps and the results. The technical code (reading, cleaning, models, plots) is in `outils_qit.py`, next to it, organised by the same sections. Its functions have French names; this notebook calls them by their English aliases (listed at the end of the file).

**How to run**: `QUICK = True` to check that everything runs (1 repeat, a few minutes), then `QUICK = False` for the results to report (10 repeats, about 25 minutes). Outputs go to `results/en/`, so they do not overwrite those of the French notebook.

> **Children's health data.** The data, and anything derived from it at the individual level, stay in `data/` (ignored by git). The `nbstripout` filter clears the notebook outputs at each commit. Never give these data to an AI tool; sections 4.3 and 8 show children one by one.

In [ ]:
import re
import subprocess
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Repository root: the first folder containing evaluate_model.py, going up from the notebook's folder.
ROOT = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "evaluate_model.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("evaluate_model.py not found: open the notebook from the FSIQ-from-ECG repository.")
sys.path.insert(0, str(ROOT))
from evaluate_model import evaluate, load_labels   # the team's shared evaluation
import outils_qit as tools                          # the technical code of this notebook
tools.set_language("en")                            # messages, tables and plots in English (computations unchanged)

In [ ]:
QUICK = True                           # True: check run (1 repeat); False: results to report (10)
TARGETS = ["QIT", "QIV", "QIP", "NCL"] # QIT = main target
T_REPEATS, K_FOLDS = 10, 5             # 10 × 5 folds for evaluation (Thursday plan)
N_ITER = 4 if QUICK else 12            # combinations tried by RandomizedSearchCV (RF, LightGBM, XGBoost, SVR)
A_PRIORI_INDICES = ["RMSSD (TD24h)", "HF (FT24h)", "SDNN (TD24h)"]   # section 4.5: chosen BEFORE seeing the results
IQ_THRESHOLD = 90                      # section 6: FSIQ < 90 versus ≥ 90
VISITS = ["M0", "M6", "M12", "M18", "M24"]      # section 7, H1: HRV measured later
TRAJECTORIES = [("M0", "M12"), ("M0", "M24")]   # section 7, H2: change in HRV between two ages
ALPHA = 0.10                           # section 8: 1 − α = 90% intervals
SEED = 42
# Cleaning and selection thresholds (correlation 0.95, top 30, PCA 90%…): at the top of outils_qit.py.

REPEATS = [1] if QUICK else list(range(1, T_REPEATS + 1))
DATA_DIR = ROOT / "data"                          # data: local only
SHARED_DIR = DATA_DIR / "cleaned"                 # cohort, folds, predictions: local only
RESULTS_DIR = Path.cwd() / "results" / "en"       # aggregated figures: model_comparison.csv, SUMMARY.md, plot
FILES = {"lin": "HRV_Lineaire_Full.csv", "nl": "HRV_NonLineaire_Full.csv", "wppsi": "Résultats WPPSI III.csv"}

# 1. Preprocessing

## 1.1 Reading the files

Three Excel exports. The HRV files have two header rows: the names, then a units row (M/F, 0/1, ms, %…) which is skipped. The WPPSI file also has two header rows, merged into "Block | Column". Empty cells may be written `""`, `" "`, `-` or `NaN`: they all become missing values.

Files are linked by the subject **number** (first column), harmonised as in `evaluate_model.py` ("012", "12.0" → "12"). Initials repeat from one child to another (see `data/pulisci_righe.py`), so the birth date is used as a check (1.2).

The targets keep the French codes of the data files:

| WPPSI score | What it measures |
|---|---|
| **QIT** | Full Scale IQ, **FSIQ** (main target) |
| **QIV** | Verbal IQ: language, knowledge |
| **QIP** | Performance IQ: non-verbal reasoning, visuo-spatial |
| **NCL** | General Language Composite: vocabulary |

Mean 100, standard deviation 15 in the population. These scores are never used as inputs. A child without all scores is removed (lost to follow-up).

In [ ]:
sources = tools.load_files(DATA_DIR, FILES)
TARGETS = [t for t in TARGETS if t in sources.scores.columns]
display(sources.scores.describe().loc[["mean", "std"]].round(1))

## 1.2 One dataset per visit

For a given visit (M0 = birth, then M6, M12, M18, M24), we keep one recording per child, the HRV indices of both files, the clinical covariates and the scores. We then remove:
- children missing from one of the 3 files, or whose birth date differs between files;
- columns that are empty at this visit: at M0, the linear file only has its **24h** indices;
- variables missing for more than 30% of the children, or near-constant (the same value for 95% of the children);
- clinical covariates that are too rare (fewer than 5 children in a class). Sex is coded 0/1.

None of these steps looks at IQ: nothing learns from the target, so they can be done before splitting the folds. Details are shown for M0, the main analysis; the other visits are used for the hypotheses of section 7.

The clinical covariates keep the French column names of the data files (perinatal / medical flags, 0/1, see `GroupA/dataset.md`):

| Column | Meaning |
|---|---|
| Sexe | sex (F = 0, M = 1) |
| Prématuré | premature birth |
| SFA | acute fetal distress |
| Réa à la naissance | resuscitation at birth |
| Pathologie neurologique | neurological condition |
| Tabac | smoking |
| Diabète | diabetes |
| HTA | high blood pressure |
| Maturation Pulm | lung maturation (antenatal corticosteroid course) |
| Dépression: prise d'IRS | depression treated with SSRIs |

In [ ]:
data = tools.build_dataset(sources, "M0", True)    # True: print the details

## 1.3 Cohort and shared folds

`evaluate_model.py` only compares models that predicted **the same children with the same folds**: `cohort.csv` (children and scores) and `folds.csv` (10 repeats of 5 folds stratified on FSIQ). The notebook writes them to `data/cleaned/` at each run: same data and same seed, hence the same folds as the French notebook. If the team ever provides its own (`prepare_data.py`), just remove the `make_folds` line.

In [ ]:
cohort_file, folds_file = SHARED_DIR / "cohort.csv", SHARED_DIR / "folds.csv"
tools.make_folds(data, cohort_file, folds_file, T_REPEATS, K_FOLDS, SEED)
data = data.sort_index(key=lambda ids: ids.astype(int))           # children sorted by number
data.rename_axis("child_id").to_csv(SHARED_DIR / "dataset_m0.csv")
labels = load_labels(cohort_file, TARGETS).loc[data.index]
folds = tools.shared_folds(folds_file, REPEATS)    # (repeat, fold, training children, test children)

X = data.drop(columns=list(sources.scores.columns))
cols_lin, cols_nl, cols_clin = ([c for c in X.columns if c.startswith(p)] for p in ("lin|", "nl|", "clin|"))
print(f"Final dataset: {len(X)} children; {len(cols_lin)} linear indices, {len(cols_nl)} non-linear, "
      f"{len(cols_clin)} clinical covariates ({', '.join(c[5:] for c in cols_clin)}); "
      f"{len(REPEATS)} × {K_FOLDS} folds")

## 1.4 Is there any signal in the variables?

Before any model, a simple benchmark: the correlation of each variable with FSIQ. With many variables, some correlate with FSIQ **by pure chance**; so we compare the strongest observed correlation with the one obtained after shuffling FSIQ between children. If the two are close, no variable is linked to FSIQ, even on its own, and no model can work miracles.

This is descriptive: choosing variables from these correlations, outside the folds, would leak information.

In [ ]:
rho, rho_chance, chance_threshold = tools.correlation_vs_chance(X, labels["QIT"])
UNIVARIATE_SIGNAL = rho.iloc[0] > chance_threshold
print(f"Strongest |Spearman| correlation with FSIQ: {rho.iloc[0]:.2f}. By pure chance, with {X.shape[1]} variables, "
      f"we get {rho_chance:.2f} on average, and up to {chance_threshold:.2f} once in 20.")
print("Conclusion:", "one correlation beats chance, a weak link may exist." if UNIVARIATE_SIGNAL else
      "nothing beats chance: no variable is linked to FSIQ, even on its own.")
print("Top 5:", rho.head(5).round(2).to_dict())

# 2. Feature selection

With about 70 HRV variables for 189 children, a model can learn the noise of the training children instead of a real link. So the variables are reduced **within each fold, on the training children only** (otherwise the test children "help" prepare the model and the results are too optimistic):

1. **Always**: fill gaps with the median, keep a single variable per group correlated above 0.95, transform each HRV index towards a normal distribution (they are very skewed).
2. **Depending on the experiment**:

| Option | Principle |
|---|---|
| `all` | no selection: the model sorts things out itself (Elastic Net sets coefficients to zero) |
| `RF selection` | the 30 most important variables of a reference random forest (supervisor's advice) |
| `PCA 90%` | the principal components that keep 90% of the variance (supervisor's advice) |

Three feature sets: `Clinical only` (the clinical baseline: what HRV should improve on), `HRV only`, `Clinical + HRV`. Below, an example of a complete pipeline: this whole block is fitted within each fold (click on a step to see its settings).

In [ ]:
FEATURE_SETS = {"Clinical only": cols_clin, "HRV only": cols_lin + cols_nl, "Clinical + HRV": cols_clin + cols_lin + cols_nl}
tools.build_pipeline(FEATURE_SETS["Clinical + HRV"], "LightGBM", "RF selection", N_ITER)

# 3. Models

Five regression models, tuned in an **inner 5-fold loop** on the training children only:
- **Elastic Net**: penalised linear regression, the reference; `ElasticNetCV` chooses the penalty strength;
- **Random Forest, LightGBM, XGBoost**: decision trees, able to capture non-linear links (the supervisor recommends LightGBM and XGBoost);
- **SVR**: kernel regression.

For the last four, `RandomizedSearchCV` tries `N_ITER` combinations drawn from the hyperparameter spaces of the Wednesday plan (`tools.SEARCH_SPACES`). An **experiment** = a model + a selection + a feature set; all of them see the same children and the same folds.

A 6th model is used in sections 5 and 6: **logistic regression**, to predict a class (premature or not, FSIQ < 90 or not).

In [ ]:
EXPERIMENTS = [                               # (model, selection, feature set)
    ("Elastic Net", "all", "Clinical only"),
    ("Elastic Net", "all", "Clinical + HRV"),
    ("Random Forest", "all", "Clinical + HRV"),
    ("LightGBM", "RF selection", "Clinical + HRV"),
    ("XGBoost", "RF selection", "Clinical + HRV"),
    ("SVR", "PCA 90%", "Clinical + HRV"),
]
EXPERIMENTS = [e for e in EXPERIMENTS if FEATURE_SETS[e[2]]]

# 4. Evaluation

Thursday protocol: **10 repeats × 5 folds** (`folds.csv`); in each fold, the model learns on 4/5 of the children and predicts the 5th. Predictions are written in the `evaluate_model.py` format (`data/cleaned/predictions/`), so they are comparable with the rest of the team. With `QUICK = True`, only the 1st repeat is run.

## 4.1 Running the experiments

In [ ]:
PRED_DIR = SHARED_DIR / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)
predictions, files = {}, {}
start = time.time()
for model, selection, feature_set in EXPERIMENTS:
    t, name = time.time(), f"{model} · {selection} · {feature_set}"
    predictions[name] = tools.cross_predictions(X[FEATURE_SETS[feature_set]], labels, folds, model, selection, N_ITER)
    files[name] = PRED_DIR / f"predictions_{re.sub(r'[^a-z0-9]+', '_', tools.normalize_text(name)).strip('_')}.csv"
    tools.write_predictions(predictions[name], folds, folds_file, cohort_file, files[name], REPEATS)
    print(f"{name}: {time.time() - t:.0f} s")
print(f"Total: {(time.time() - start) / 60:.1f} min")

## 4.2 Model comparison

One measurement per test fold, summarised as mean ± standard deviation (MAE, MSE, RMSE, R²) and written to `results/en/model_comparison.csv`. The **baseline** predicts, in each fold, the mean score of the training children: this is the floor to beat. The **gain** is what the model gains in MAE over this baseline; its 95% confidence interval comes from `evaluate_model.py` (bootstrap over children). **If the gain's interval contains 0, the improvement is not demonstrated.**

In [ ]:
per_fold = tools.per_fold_metrics(predictions, labels, folds)
model_comparison = tools.comparison_by_model(per_fold)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_comparison.round(4).to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
team_summary, _ = evaluate(folds_file, cohort_file, [str(f) for f in files.values()],
                           names=list(files), targets=TARGETS, verbose=False)
display(tools.results_table(model_comparison, team_summary, TARGETS))
tools.plot_gains(per_fold, TARGETS, RESULTS_DIR / "model_comparison.png")
models_only = per_fold[per_fold.model != tools.REFERENCE]
BEST = {t: models_only[models_only.target == t].groupby("model")["MAE"].mean().idxmin() for t in TARGETS}

## 4.3 Predicted versus observed (FSIQ, best model)

One point = one child, predicted when it was in the test fold (1st repeat). **Individual-level plot: do not share.** If the points form a horizontal band around the mean FSIQ, the model predicts almost the same thing for everyone: it found no signal.

In [ ]:
BEST_MODEL = BEST["QIT"]
best_model, best_selection, best_set = next(e for e in EXPERIMENTS if " · ".join(e) == BEST_MODEL)
true, pred = labels["QIT"], predictions[BEST_MODEL]["QIT"].loc[REPEATS[0]]
tools.plot_predicted_observed(true, pred, BEST_MODEL)
error = (true - pred).abs()
print(f"Standard deviation of the predictions: {pred.std():.1f} points (that of the true FSIQ: {true.std():.1f}). "
      f"MAE for FSIQ < {IQ_THRESHOLD}: {error[true < IQ_THRESHOLD].mean():.1f} points, "
      f"for the others: {error[true >= IQ_THRESHOLD].mean():.1f}.")

## 4.4 Why it adds nothing, and when it would have added something

**Why.** Three reasons, from the most to the least important:
1. **There is no link in the variables** (section 1.4): no variable correlates with FSIQ more than chance does. A model does not create a missing signal; it can only combine those that exist.
2. **With 189 children, looking for a signal that does not exist is costly.** Each coefficient learned from noise adds error. Elastic Net senses this and sets almost everything to zero (it falls back on the mean); trees, being more flexible, learn the noise and do **worse** than the mean (section 4.2).
3. **IQ at age 3 depends on many things absent here**: family environment, stimulation, socio-economic status, events of the first three years. HRV at birth sees none of this.

**When it would have added something.** To find out, we build a **fake IQ** that truly depends on 5 HRV indices, with a known strength: it explains 0%, 5%, 10%, 20% or 30% of the IQ variance (the "planted" R²), the rest being noise. We then run exactly the same method (Elastic Net, Clinical + HRV, same children). This shows from what link strength our method would have detected it, and places our real result on that scale. Section 4.5 then bounds this link with a more sensitive test.

In [ ]:
kept, total = tools.kept_variables(X[FEATURE_SETS["Clinical + HRV"]], labels["QIT"], folds), len(FEATURE_SETS["Clinical + HRV"])
print(f"Elastic Net (Clinical + HRV, FSIQ) keeps on average {kept:.1f} variables out of {total} per fold"
      + (": almost none, it falls back on the mean for lack of a reliable link." if kept < 0.1 * total else "."))

simulation = tools.detection_simulation(X, FEATURE_SETS["Clinical + HRV"], cols_lin + cols_nl, labels["QIT"],
                                        [0.0, 0.05, 0.10, 0.20, 0.30], len(REPEATS), K_FOLDS, SEED)
display(simulation.round(2))
en_name = "Elastic Net · all · Clinical + HRV"
real = team_summary.set_index(["target", "model"]).loc[("QIT", en_name)] if en_name in files else None
tools.plot_simulation(simulation, real)

detected = simulation.index[simulation["CI low"] > 0]
DETECTION_THRESHOLD = detected.min() if len(detected) else None
print((f"With these {len(X)} children, the method detects a link as soon as it explains about "
       f"{DETECTION_THRESHOLD:.0%} of the IQ variance." if DETECTION_THRESHOLD is not None else
       "No simulated level is detected with this repeat: run again with QUICK = False.")
      + (" Our real gain is not demonstrated: if neonatal HRV explains part of IQ at age 3, "
         "that part is smaller than this threshold." if real is not None and real.gain_ci_low <= 0 else ""))

## 4.5 What share of IQ can HRV explain, at most?

According to section 4.4, prediction only sees a link from about 20% of explained variance. "We saw nothing" therefore leaves open the possibility of a weaker link, of 5 or 10%. To bound it, we change the question: instead of **predicting** IQ with about a hundred variables (and paying the price of searching among all of them), we **test** a few indices **chosen in advance**. With 4 targeted tests, the method is much more sensitive.

The indices are chosen from the literature, without looking at our results (`A_PRIORI_INDICES`):
- **RMSSD** and **HF** (24h): markers of **vagal tone**, the brake that the parasympathetic system puts on the heart. Porges' polyvagal theory links newborn vagal tone to later development (for example Doussard-Roosevelt et al., 1997, in preterm infants);
- **SDNN** (24h): overall variability;
- the **1st principal component** of all HRV indices: a summary of the whole set, computed without looking at IQ.

For each one, we compute the Spearman correlation with FSIQ, raw and then **adjusted** for the clinical covariates (what HRV would add beyond prematurity, sex…), with its confidence interval. The intervals are widened to account for the 4 tests (Bonferroni correction): the 4 bounds hold **together** with 95% confidence.

**How to read it**:
- if an interval excludes 0, the link is demonstrated, even if weak;
- otherwise, the squared upper bound gives the **largest share of FSIQ variance compatible with our data** (variance of ranks, which is almost the same). "HRV explains at most X%" is a result, stronger than "we saw nothing".

Raw and adjusted correlations form two families of 4 tests, each at 95% confidence. The main question is the raw correlation; the adjusted one complements it. A raw link that disappears after adjustment probably goes through the clinical covariates (prematurity changes both HRV and IQ), not through HRV itself.

*Checked on 8,000 simulated datasets of 189 children where the truth is known (`python verifier_bornes.py`, about 1 minute):*
- *with no link at all, at most 4% false positives;*
- *a link at the announced threshold is detected 77 times out of 100;*
- *a fake link created by prematurity is removed by the adjustment (4% adjusted false positives, versus 32% raw).*

> Do not change `A_PRIORI_INDICES` after seeing the results: picking the indices that work best would invalidate the test (the same information leak as in section 1.4).

In [ ]:
cols_hrv = cols_lin + cols_nl
chosen = {name: tools.find_column(X[cols_hrv], name) for name in A_PRIORI_INDICES}
if None in chosen.values():
    print("Indices not found, ignored:", [n for n, c in chosen.items() if c is None])
indices = X[[c for c in chosen.values() if c is not None]].rename(columns=lambda c: c.split("|", 1)[1])
indices["1st HRV component"], share_pc1 = tools.first_component(X, cols_hrv)
print(f"The 1st principal component summarises {share_pc1:.0%} of the variance of the {len(cols_hrv)} HRV indices.")

bounds = tools.correlation_bounds(indices, labels["QIT"], X[cols_clin])
DETECTABLE_VARIANCE = tools.detectable_variance(len(X), indices.shape[1])
display(tools.bounds_table(bounds))
tools.plot_bounds(bounds, np.sqrt(DETECTABLE_VARIANCE))

BOUND, ADJUSTED_BOUND = bounds["max variance"].max(), bounds["adjusted max variance"].max()
DEMONSTRATED = ([f"{i} (raw)" for i in bounds.index[bounds["demonstrated"] == "yes"]]
                + [f"{i} (adjusted)" for i in bounds.index[bounds["adjusted demonstrated"] == "yes"]])
print(f"Sensitivity: with {len(X)} children and {indices.shape[1]} tests, a link explaining {DETECTABLE_VARIANCE:.0%} "
      "of the FSIQ variance is detected 8 times out of 10"
      + (f" (versus {DETECTION_THRESHOLD:.0%} for prediction, section 4.4)." if DETECTION_THRESHOLD is not None else "."))
RAW_ONLY = list(bounds.index[(bounds["demonstrated"] == "yes") & (bounds["adjusted demonstrated"] == "no")])
print((f"Link demonstrated for: {', '.join(DEMONSTRATED)}. " if DEMONSTRATED else "No link demonstrated. ")
      + f"With 95% confidence, none of these indices explains more than {BOUND:.0%} of the FSIQ variance "
      f"({ADJUSTED_BOUND:.0%} beyond the clinical covariates).")
if RAW_ONLY:
    print(f"Warning: the link of {', '.join(RAW_ONLY)} disappears after adjusting for the clinical covariates. "
          "It probably goes through them (for example prematurity, which changes both HRV and IQ), "
          "not through HRV itself.")

# 5. Positive control: does HRV recognise prematurity?

A null result can have two very different causes: **there is no link**, or **our method (or our data) cannot see one**. The positive control tells them apart: we ask the same method a question whose answer we **know**.

A newborn's HRV is known to depend strongly on maturity: a child born prematurely has a less mature autonomic nervous system, hence a different HRV. So we ask the HRV indices alone to guess whether the child is premature.

- If HRV does recognise prematurity (AUC clearly above 0.5), **the HRV data contain physiological information and the method can extract it**. The absence of a link with IQ then becomes a real result, not a data or code problem.
- If it fails, the data should be questioned first (recording quality, definition of prematurity in the file).

The **AUC** reads as follows: 0.5 = the model does no better than a coin toss; 1 = it separates the two groups perfectly; above 0.7, discrimination is generally considered good.

In [ ]:
control = None
col_prem = "clin|Prématuré"                                       # column name in the data file
if col_prem not in X or X[col_prem].nunique() != 2:
    print("Prematurity not available: no positive control.")
else:
    known = X[col_prem].dropna()
    premature = (known == known.max()).astype(int)               # 1 = premature (0/1 coding of the file)
    print(f"{premature.sum()} premature children out of {len(premature)} with this information")
    if min(premature.sum(), (1 - premature).sum()) < 10:
        print("Fewer than 10 children in one of the two groups: the control is too fragile to conclude.")
    else:
        folds_prem = tools.stratified_folds(premature, len(REPEATS), K_FOLDS, SEED)
        P_prem = tools.cross_predictions(X.loc[premature.index, cols_lin + cols_nl], premature, folds_prem, "Logistic")
        control = tools.classification_summary(premature, P_prem)
        display(pd.DataFrame([control], index=["HRV only → prematurity"]).round(2))
        print("Verdict:", "HRV does recognise prematurity: the data and the method are able to see a link."
              if control["AUC CI low"] > 0.5 else
              "HRV does not recognise prematurity in a demonstrated way: question the data first.")

# 6. Classification: spotting children with FSIQ < 90

Predicting IQ to the point may be asking too much. A more modest question, and more useful clinically: **can we spot the children whose FSIQ will be below 90** (about 1 child in 5), to follow them more closely? A signal limited to low IQs could go unnoticed in the regression.

**The imbalance trap.** With about 80% of children above 90, a "model" that always answers "≥ 90" has 80% accuracy while knowing nothing. Accuracy alone is therefore misleading. We look instead at:
- the **AUC**: 0.5 = chance, 1 = perfect;
- **sensitivity** (share of children < 90 spotted) and **specificity** (share of children ≥ 90 correctly left aside);
- **balanced accuracy** (mean of the two: 0.5 = chance);
- **Cohen's kappa** (agreement beyond chance: 0 = chance, 1 = perfect).

The model is a penalised logistic regression, both classes counting equally (`class_weight="balanced"`). It is evaluated on the same folds as the regression. The "always ≥ 90" row shows the accuracy trap.

In [ ]:
low = (labels["QIT"] < IQ_THRESHOLD).astype(int)
print(f"{low.sum()} children out of {len(low)} ({low.mean():.0%}) have FSIQ < {IQ_THRESHOLD}")
classification = {f"always '≥ {IQ_THRESHOLD}'": {"AUC": 0.5, "AUC CI low": 0.5, "AUC CI high": 0.5,
                                                 "sensitivity": 0.0, "specificity": 1.0,
                                                 "accuracy": 1 - low.mean(), "balanced accuracy": 0.5, "kappa": 0.0}}
for feature_set in ["Clinical only", "Clinical + HRV"]:
    if FEATURE_SETS[feature_set]:
        P = tools.cross_predictions(X[FEATURE_SETS[feature_set]], low, folds, "Logistic")
        classification[f"Logistic · {feature_set}"] = tools.classification_summary(low, P)
classification = pd.DataFrame(classification).T
display(classification.round(2))

# 7. Hypotheses: later HRV and trajectories

HRV at birth does not predict IQ at age 3. Two hypotheses to test before concluding that HRV plays no role:

- **H1: HRV measured later predicts better.** At 6, 12, 18 or 24 months, the autonomic nervous system is more mature and more stable, and the measurement is closer to the assessment at age 3.
- **H2: what matters is the change, not the value at a given age.** A child whose HRV matures quickly between birth and 12 (or 24) months might develop better. The variables are then the differences in HRV indices between the two ages.

**Test**: for each analysis, Elastic Net with the clinical covariates and the HRV indices, compared with the mean FSIQ, in cross-validation on the children who have the necessary recordings. A hypothesis is supported if the confidence interval of the gain is above 0.

**Caveats**: the children differ from one analysis to another (not all have every recording), so the results cannot be compared point by point. And with 7 tests, an isolated positive result may be due to chance: it would need to be confirmed on other data before believing it.

In [ ]:
rows = []


def test(d, analysis):
    """Elastic Net (all variables of d except FSIQ) against the mean FSIQ, in cross-validation."""
    columns = [c for c in d.columns if c.startswith(("lin|", "nl|", "evol|", "clin|"))]
    if len(d) < 50 or not columns:
        print(f"{analysis}: only {len(d)} children, analysis skipped")
        return
    folds_h = tools.stratified_folds(d["QIT"], len(REPEATS), K_FOLDS, SEED)
    P = tools.cross_predictions(d[columns], d["QIT"], folds_h)
    rows.append({"analysis": analysis, "variables": len(columns), **tools.gain_vs_mean(d["QIT"], P, folds_h)})


datasets = {v: tools.build_dataset(sources, v) for v in VISITS}
for visit, d in datasets.items():
    test(d, f"H1 · HRV at {visit}")
for before, after in TRAJECTORIES:
    d1, d2 = datasets[before], datasets[after]
    children = d1.index.intersection(d2.index)
    common = [c for c in d1.columns if c.startswith(("lin|", "nl|")) and c in d2.columns]
    change = (d2.loc[children, common] - d1.loc[children, common]).add_prefix("evol|")
    test(change.join(d1.loc[children, [c for c in d1.columns if c.startswith("clin|")] + ["QIT"]]),
         f"H2 · change {before} → {after}")

hypotheses = pd.DataFrame(rows).set_index("analysis")
hypotheses["supported"] = np.where(hypotheses["CI low"] > 0, "yes", "no")
display(hypotheses.round(2))
tools.plot_hypotheses(hypotheses)

# 8. Conformal prediction (FSIQ)

A predicted FSIQ alone says nothing about its margin of error. **Conformal prediction** gives an interval [prediction − q ; prediction + q] that contains the true FSIQ with a probability of about 90%, whatever the model. The half-width q is measured on the model's errors on children **it has not seen**: for each test fold, an inner cross-validation on the training children only. Test children are never used to calibrate their own interval, which makes it possible to check the coverage honestly.

If the interval is no narrower than the one built around the plain mean FSIQ, the model adds nothing for an individual child. **Individual-level plot: do not share.**

In [ ]:
r1 = REPEATS[0]
pred_r1 = predictions[BEST_MODEL]["QIT"].loc[r1]
conformal = tools.conformal_intervals(X[FEATURE_SETS[best_set]], labels["QIT"], pred_r1, [f for f in folds if f[0] == r1],
                                      ALPHA, best_model, best_selection, N_ITER)
coverage = (np.abs(conformal["true"] - conformal["pred"]) <= conformal["q"]).mean()
coverage_ref = (np.abs(conformal["true"] - conformal["mean"]) <= conformal["q_ref"]).mean()
q_final = tools.conformal_quantile((labels["QIT"] - pred_r1).abs().to_numpy(), ALPHA)
display(pd.DataFrame({"observed coverage": [f"{coverage:.0%}", f"{coverage_ref:.0%}"],
                      "half-width (FSIQ points)": [f"± {conformal['q'].mean():.1f}", f"± {conformal['q_ref'].mean():.1f}"]},
                     index=[BEST_MODEL, "around the mean FSIQ"]))
print(f"For a new child: predicted FSIQ ± {q_final:.1f} points, with a probability of about {1 - ALPHA:.0%} "
      "of containing their true FSIQ.")
tools.plot_intervals(conformal, IQ_THRESHOLD, ALPHA)

# 9. Feature explanation

The importance of a variable is measured as follows: shuffle its values among the test children and see how much performance drops. If it drops a lot, the model relied on that variable.

- **For IQ**, no model does better than the mean (section 4): the importance of its variables would only measure noise, and Elastic Net sets almost all its coefficients to zero (section 4.4). The honest answer to "which variable matters for IQ?" is therefore: **none, with these data**.
- **For prematurity** (positive control, section 5), HRV carries a real signal: that is where explanation makes sense. The plot shows the HRV indices that make it possible to recognise a premature child.

An important variable is **associated** with the outcome in this cohort; it is not a **cause**. "What should be done for a child predicted below 90?" is a causal question: a predictive model can at best help identify children to follow up, and interventions are a matter for clinical studies.

In [ ]:
if control is None:
    print("No usable positive control: no explanation to show.")
else:
    importances = tools.permutation_importances(X.loc[premature.index, cols_lin + cols_nl], premature,
                                                [f for f in folds_prem if f[0] == 1])
    tools.plot_importances(importances, "HRV indices that recognise prematurity")

# 10. Summary for the supervisor

Generated from the results above (aggregated figures only) and saved to `results/en/SUMMARY.md`.

In [ ]:
team = team_summary.set_index(["target", "model"])
comp = model_comparison.set_index(["target", "model"])
demonstrated = [t for t in TARGETS if team.loc[(t, BEST[t])].gain_ci_low > 0]
mae_qit = comp.xs("QIT")["mae_mean"]
alone, added = "Elastic Net · all · Clinical only", "Elastic Net · all · Clinical + HRV"
best_classif = classification.iloc[1:]["AUC"].idxmax()
classif = classification.loc[best_classif]
prematurity_found = control is not None and control["AUC CI low"] > 0.5


def regression_row(target):
    m, e = BEST[target], team.loc[(target, BEST[target])]
    s, ref = comp.loc[(target, m)], comp.loc[(target, tools.REFERENCE)]
    return (f"| {target} | {m} | {s.mae_mean:.2f} ± {s.mae_std:.2f} | {s.r2_mean:.3f} | {ref.mae_mean:.2f} | "
            f"{e.MAE_gain_vs_baseline:+.2f} {tools.ci(e.gain_ci_low, e.gain_ci_high)} "
            f"({'demonstrated' if e.gain_ci_low > 0 else 'not demonstrated'}) |")


SUMMARY = "\n".join([
    "# Predicting IQ at age 3 from neonatal HRV: summary",
    f"*Group A · Jade · generated on {date.today():%d/%m/%Y} by `01_prediction_qit_en.ipynb` · "
    + ("quick mode (1 repeat), provisional figures*" if QUICK else f"{T_REPEATS} × {K_FOLDS} folds*"),
    "",
    "## Conclusion",
    (f"**Gain over the mean score demonstrated for: {', '.join(demonstrated)}.** " if demonstrated else
     "**No model does better than predicting the mean score, for any target.** ")
    + (f"Yet HRV does recognise prematurity (AUC {control['AUC']:.2f}): the data and the method can see a link "
       "when there is one." if prematurity_found else ""),
    "",
    "## Data and method",
    f"- {len(X)} children of the AuBE cohort (recording at birth, complete WPPSI-III scores); "
    f"{len(cols_lin)} linear HRV indices, {len(cols_nl)} non-linear, {len(cols_clin)} clinical covariates.",
    f"- Models: {', '.join(sorted({e[0] for e in EXPERIMENTS}))}; feature reduction within each fold; "
    f"evaluation on the team's shared folds ({len(REPEATS)} × {K_FOLDS}, `evaluate_model.py`).",
    "- Targets (French codes of the data): QIT = Full Scale IQ, QIV = Verbal IQ, QIP = Performance IQ, "
    "NCL = General Language Composite.",
    "",
    "## Regression (best model per target)",
    "| Target | Model | MAE ± std | R² | MAE of the mean | Gain vs mean [95% CI] |",
    "|---|---|---|---|---|---|",
    *[regression_row(t) for t in TARGETS],
    "",
    *([f"- Adding HRV to the clinical model (Elastic Net, FSIQ): MAE goes from {mae_qit[alone]:.2f} to "
       f"{mae_qit[added]:.2f} ({'better' if mae_qit[added] < mae_qit[alone] else 'worse'})."]
      if alone in mae_qit and added in mae_qit else []),
    f"- Signal in the variables: the strongest correlation with FSIQ is {rho.iloc[0]:.2f}, versus "
    f"{rho_chance:.2f} on average by chance (up to {chance_threshold:.2f})"
    + (": one correlation beats chance." if UNIVARIATE_SIGNAL else ": nothing beats chance, no link to learn."),
    *([f"- Detection limit (simulation): with these children, the method sees a link as soon as it explains about "
       f"{DETECTION_THRESHOLD:.0%} of the IQ variance" + ("." if demonstrated else "; if HRV explains part of it, that part is smaller.")]
      if DETECTION_THRESHOLD is not None else []),
    f"- Bound (indices chosen in advance: {', '.join(indices.columns)}): "
    + (f"link demonstrated for {', '.join(DEMONSTRATED)}; " if DEMONSTRATED else "no link demonstrated; ")
    + f"none explains more than {BOUND:.0%} of the FSIQ variance ({ADJUSTED_BOUND:.0%} beyond the clinical "
    f"covariates), simultaneous 95% intervals; a link of {DETECTABLE_VARIANCE:.0%} would have been detected 8 times out of 10"
    + (f". The raw link of {', '.join(RAW_ONLY)} disappears after adjusting for the clinical covariates."
       if RAW_ONLY else "."),
    f"- {1 - ALPHA:.0%} prediction interval (conformal prediction): ± {q_final:.1f} points, versus "
    f"± {conformal['q_ref'].mean():.1f} around the mean FSIQ.",
    "",
    "## Positive control, classification, hypotheses",
    (f"- Positive control: HRV only → prematurity, AUC {control['AUC']:.2f} "
     f"{tools.ci(control['AUC CI low'], control['AUC CI high'])}." if control is not None else
     "- Positive control: prematurity not available or too rare."),
    f"- Classification FSIQ < {IQ_THRESHOLD} ({low.sum()} children): best AUC {classif['AUC']:.2f} "
    f"{tools.ci(classif['AUC CI low'], classif['AUC CI high'])} ({best_classif}), balanced accuracy "
    f"{classif['balanced accuracy']:.2f}, kappa {classif['kappa']:.2f}.",
    *[f"- {a} ({int(h.children)} children): gain {h.gain:+.2f} {tools.ci(h['CI low'], h['CI high'])} "
      f"({'supported' if h['CI low'] > 0 else 'not supported'})." for a, h in hypotheses.iterrows()],
    f"- With {len(hypotheses)} hypotheses tested, an isolated supported hypothesis may be due to chance: to be confirmed.",
    "",
    "## Limitations",
    "- About 190 children; no external cohort; IQ at age 3 depends on many factors not measured here "
    "(family environment, socio-economic status).",
    "- Association is not causation: a predictive model does not say how to improve a child's IQ.",
    "",
    "## Next steps",
    "- Variables derived from the raw data (RR files, ECG signal): Group B's work.",
    "- Confirm any hypothesis supported above on another cohort.",
])
(RESULTS_DIR / "SUMMARY.md").write_text(SUMMARY + "\n", encoding="utf-8")
display(Markdown(SUMMARY))

# 11. Before committing

The plan asks to check that no health data is tracked by git. `results/` only contains aggregated figures; `data/` stays local.

In [ ]:
tracked = subprocess.run(["git", "ls-files"], cwd=ROOT, capture_output=True, text=True).stdout.splitlines()
suspects = [f for f in tracked if re.search(r"\.(csv|xlsx?|mat|edf|zip|pkl|joblib)$", f, re.I)]
print("Data files tracked by git:", suspects or "none")
print("This notebook's outputs are cleared at commit by nbstripout; without that filter, use 'Restart & Clear Output'.")